# Imbalanced Fraud Detection Dataset

# Columns:
- transaction_id, timestamp, customer_id, account_age_days
- home_country, merchant_id, merchant_country, merchant_category
- device_type, transaction_channel, amount, distance_from_home
- is_cross_border, is_subscription, subscription_name
- is_fraud, fraud_source, fraud_scenario

# Fraud sources:
- none: normal transaction
- account_takeover: fraud on an established customer account
- stolen_card: fraud on an existing customer payment instrument
- new_account_fraud: fraud from a newly created account

# Fraud scenarios:
- none: normal transaction
- ping: 3 to 5 tiny online transactions used to test a card
- high_value_extrication: one large purchase at a high-risk merchant
- velocity_attack: 10 or more transactions at the same merchant within 5 minutes, happened to my uncle once sadly

# INFO:
- Customers repeat across many transactions
- account_age_days is derived from the customer signup date and transaction timestamp
- Normal purchase amounts follow a log-normal distribution
- Normal distances follow an exponential distribution
- Transaction times follow daily shopping patterns, with low activity overnight
- Subscriptions create legitimate recurring online behavior
- Fraud is generated separately, appended, shuffled, and controlled by target_fraud_rate
- fraud_source and fraud_scenario are ground-truth labels and so pls before model training


In [1]:
import os
import uuid

import numpy as np
import pandas as pd

In [2]:
random_seed = 42 # answer to all things in the universe xD

In [3]:
COUNTRIES = ["NL", "BE", "DE", "GB", "FR", "ES", "IN", "SG", "CA", "AU"]

COUNTRY_PROBABILITIES = np.array([0.62, 0.12, 0.07, 0.05, 0.035, 0.025, 0.03, 0.015, 0.025, 0.01])

CROSS_BORDER_MERCHANT_PROBABILITIES = np.array([0.00, 0.22, 0.22, 0.14, 0.13, 0.09, 0.09, 0.05, 0.04, 0.02])

DEVICE_TYPES = ["android", "iphone", "desktop", "tablet", "laptop", "macbook"]

DEVICE_TYPE_PROBABILITIES = np.array([0.34, 0.30, 0.15, 0.07, 0.09, 0.05])

FRAUD_DEVICE_TYPE_PROBABILITIES = np.array([0.42, 0.35, 0.08, 0.04, 0.07, 0.04])

MERCHANT_CATEGORIES = [
    "Groceries",
    "Food & Dining",
    "Public Transport",
    "Bike & Mobility",
    "Fuel",
    "Retail",
    "Electronics",
    "Travel",
    "Entertainment",
    "Digital Goods",
    "Health",
    "Jewelry",
    "Luxury Retail",
]

MERCHANT_CATEGORY_PROBABILITIES = np.array([
    0.22,
    0.15,
    0.07,
    0.04,
    0.035,
    0.16,
    0.07,
    0.04,
    0.08,
    0.07,
    0.045,
    0.01,
    0.01,
])

CATEGORY_AMOUNT_MULTIPLIERS = {
    "Groceries": 0.9,
    "Food & Dining": 0.6,
    "Public Transport": 0.45,
    "Bike & Mobility": 0.85,
    "Fuel": 0.95,
    "Retail": 1.1,
    "Electronics": 3.2,
    "Travel": 3.8,
    "Entertainment": 0.9,
    "Digital Goods": 0.5,
    "Health": 1.4,
    "Jewelry": 5.0,
    "Luxury Retail": 4.5,
}

CATEGORY_ONLINE_PROBABILITIES = {
    "Groceries": 0.08,
    "Food & Dining": 0.12,
    "Public Transport": 0.60,
    "Bike & Mobility": 0.55,
    "Fuel": 0.01,
    "Retail": 0.35,
    "Electronics": 0.55,
    "Travel": 0.75,
    "Entertainment": 0.40,
    "Digital Goods": 0.96,
    "Health": 0.20,
    "Jewelry": 0.30,
    "Luxury Retail": 0.45,
}

CREDIT_CARD_ONLINE_PROBABILITIES = {
    "Groceries": 0.06,
    "Food & Dining": 0.08,
    "Public Transport": 0.08,
    "Bike & Mobility": 0.10,
    "Fuel": 0.05,
    "Retail": 0.12,
    "Electronics": 0.18,
    "Travel": 0.35,
    "Entertainment": 0.16,
    "Digital Goods": 0.22,
    "Health": 0.10,
    "Jewelry": 0.28,
    "Luxury Retail": 0.32,
}

TRANSACTION_CHANNELS = ["pos_pin", "ideal_online", "credit_card_online", "atm_cash"]

ONLINE_CHANNELS = ["ideal_online", "credit_card_online"]

SUBSCRIPTION_PRODUCTS = [
    {"subscription_name": "Netflix", "merchant_category": "Entertainment", "base_amount": 15.49},
    {"subscription_name": "Spotify", "merchant_category": "Entertainment", "base_amount": 10.99},
    {"subscription_name": "Videoland", "merchant_category": "Entertainment", "base_amount": 10.99},
    {"subscription_name": "NPO Plus", "merchant_category": "Entertainment", "base_amount": 2.95},
    {"subscription_name": "NS Flex", "merchant_category": "Public Transport", "base_amount": 5.60},
    {"subscription_name": "Swapfiets", "merchant_category": "Bike & Mobility", "base_amount": 19.90},
    {"subscription_name": "Basic-Fit", "merchant_category": "Health", "base_amount": 29.99},
    {"subscription_name": "Bol Select", "merchant_category": "Retail", "base_amount": 12.99},
    {"subscription_name": "Cloud Storage", "merchant_category": "Digital Goods", "base_amount": 9.99},
    {"subscription_name": "News Subscription", "merchant_category": "Digital Goods", "base_amount": 7.99},
    {"subscription_name": "Meal Kit", "merchant_category": "Food & Dining", "base_amount": 59.99},
]

OUTPUT_COLUMNS = [
    "transaction_id",
    "timestamp",
    "customer_id",
    "account_age_days",
    "home_country",
    "merchant_id",
    "merchant_country",
    "merchant_category",
    "device_type",
    "transaction_channel",
    "amount",
    "distance_from_home",
    "is_cross_border",
    "is_subscription",
    "subscription_name",
    "is_fraud",
    "fraud_source",
    "fraud_scenario",
]

In [4]:
def choose_other_country(home_country, rng):
    probabilities = CROSS_BORDER_MERCHANT_PROBABILITIES.copy()

    home_country_position = COUNTRIES.index(home_country)
    probabilities[home_country_position] = 0.0
    probabilities = probabilities / probabilities.sum()

    return rng.choice(COUNTRIES, p=probabilities)

def make_merchant_id(merchant_category, merchant_country, rng):
    clean_category = merchant_category.upper().replace("&", "AND").replace(" ", "_")
    merchant_number = rng.integers(1, 200)
    return f"{merchant_country}_{clean_category}_{merchant_number:03d}"


def make_transaction_ids(n_transactions, seed=random_seed):
    return [
        str(uuid.uuid5(uuid.NAMESPACE_DNS, f"beegbank_fraud_{seed}_{row_number}"))
        for row_number in range(n_transactions)
    ]


def get_account_age_days(timestamp, signup_date):
    age_days = (pd.Timestamp(timestamp).normalize() - pd.Timestamp(signup_date).normalize()).days
    return max(0, int(age_days))

In [5]:
def sample_normal_hours(n_samples, rng):
    hours = []

    while len(hours) < n_samples:
        n_needed = n_samples - len(hours)

        peak_names = rng.choice(
            ["midday", "evening", "other"],
            size=n_needed * 2,
            p=[0.45, 0.45, 0.10],
        )

        sampled_hours = np.empty(n_needed * 2)

        midday_mask = peak_names == "midday"
        evening_mask = peak_names == "evening"
        other_mask = peak_names == "other"

        sampled_hours[midday_mask] = rng.normal(12.5, 2.8, midday_mask.sum())
        sampled_hours[evening_mask] = rng.normal(18.5, 2.4, evening_mask.sum())
        sampled_hours[other_mask] = rng.uniform(6.0, 23.0, other_mask.sum())

        sampled_hours = sampled_hours[(sampled_hours >= 0.0) & (sampled_hours < 24.0)]
        hours.extend(sampled_hours[:n_needed])

    return np.array(hours[:n_samples])


def sample_normal_timestamps(start, n_days, n_samples, rng):
    start_timestamp = pd.Timestamp(start)

    day_offsets = rng.integers(0, n_days, size=n_samples)
    hours = sample_normal_hours(n_samples, rng)
    minutes = rng.integers(0, 60, size=n_samples)
    seconds = rng.integers(0, 60, size=n_samples)

    return (
        start_timestamp
        + pd.to_timedelta(day_offsets, unit="D")
        + pd.to_timedelta(hours, unit="h")
        + pd.to_timedelta(minutes, unit="m")
        + pd.to_timedelta(seconds, unit="s")
    )


def sample_late_night_timestamp(start, n_days, rng):
    start_timestamp = pd.Timestamp(start)

    day_offset = rng.integers(0, n_days)
    hour = rng.uniform(0.0, 4.0)
    minute = rng.integers(0, 60)
    second = rng.integers(0, 60)

    return (
        start_timestamp
        + pd.to_timedelta(day_offset, unit="D")
        + pd.to_timedelta(hour, unit="h")
        + pd.to_timedelta(minute, unit="m")
        + pd.to_timedelta(second, unit="s")
    )

In [6]:
def make_customers(n_customers, start, seed=random_seed):
    rng = np.random.default_rng(seed)

    customer_ids = [f"CUST_{customer_number:05d}" for customer_number in range(1, n_customers + 1)]

    age_groups = rng.choice(
        ["new", "regular", "mature"],
        size=n_customers,
        p=[0.08, 0.34, 0.58],
    )

    account_age_at_start = np.empty(n_customers, dtype=int)
    account_age_at_start[age_groups == "new"] = rng.integers(1, 90, size=(age_groups == "new").sum())
    account_age_at_start[age_groups == "regular"] = rng.integers(90, 730, size=(age_groups == "regular").sum())
    account_age_at_start[age_groups == "mature"] = rng.integers(730, 3650, size=(age_groups == "mature").sum())

    start_timestamp = pd.Timestamp(start)
    signup_dates = start_timestamp - pd.to_timedelta(account_age_at_start, unit="D")

    customers = pd.DataFrame({
        "customer_id": customer_ids,
        "signup_date": signup_dates,
        "home_country": rng.choice(COUNTRIES, size=n_customers, p=COUNTRY_PROBABILITIES),
        "primary_device_type": rng.choice(DEVICE_TYPES, size=n_customers, p=DEVICE_TYPE_PROBABILITIES),
        "normal_spend_level": rng.lognormal(mean=0.0, sigma=0.45, size=n_customers),
        "normal_travel_radius": rng.gamma(shape=2.0, scale=5.0, size=n_customers) + 1.0,
        "activity_level": rng.gamma(shape=2.0, scale=1.0, size=n_customers),
    })

    return customers

In [7]:
def choose_transaction_channels(merchant_categories, rng):
    channels = []

    for merchant_category in merchant_categories:
        if rng.random() < 0.02:
            channels.append("atm_cash")
            continue

        online_probability = CATEGORY_ONLINE_PROBABILITIES[merchant_category]

        if rng.random() < online_probability:
            credit_card_probability = CREDIT_CARD_ONLINE_PROBABILITIES[merchant_category]

            if rng.random() < credit_card_probability:
                channels.append("credit_card_online")
            else:
                channels.append("ideal_online")
        else:
            channels.append("pos_pin")

    return np.array(channels)


def choose_device_types(primary_device_types, rng):
    device_types = []

    for primary_device_type in primary_device_types:
        if rng.random() < 0.88:
            device_types.append(primary_device_type)
        else:
            other_devices = [device_type for device_type in DEVICE_TYPES if device_type != primary_device_type]
            device_types.append(rng.choice(other_devices))

    return np.array(device_types)


def sample_normal_amounts(merchant_categories, spend_levels, rng):
    amounts = []

    for merchant_category, spend_level in zip(merchant_categories, spend_levels):
        category_multiplier = CATEGORY_AMOUNT_MULTIPLIERS[merchant_category]
        median_amount = 28.0 * spend_level * category_multiplier
        amount = rng.lognormal(mean=np.log(median_amount), sigma=0.75)
        amounts.append(amount)

    return np.round(np.clip(amounts, 1.0, 5000.0), 2)


def sample_normal_cross_border_flags(merchant_categories, transaction_channels, rng):
    flags = []

    for merchant_category, transaction_channel in zip(merchant_categories, transaction_channels):
        probability = 0.012

        if transaction_channel == "ideal_online":
            probability += 0.010

        if transaction_channel == "credit_card_online":
            probability += 0.055

        if merchant_category == "Travel":
            probability += 0.120

        if merchant_category in ["Digital Goods", "Electronics", "Luxury Retail", "Jewelry"]:
            probability += 0.025

        flags.append(rng.random() < probability)

    return np.array(flags, dtype=bool)


def sample_distances_from_home(normal_travel_radii, is_cross_border, transaction_channels, rng):
    distances = rng.exponential(scale=normal_travel_radii)

    online_mask = np.isin(transaction_channels, ONLINE_CHANNELS)
    distances[online_mask] = rng.exponential(scale=35.0, size=online_mask.sum())

    atm_mask = transaction_channels == "atm_cash"
    distances[atm_mask] = rng.exponential(scale=2.5, size=atm_mask.sum())

    cross_border_mask = is_cross_border == True
    distances[cross_border_mask] = rng.lognormal(mean=np.log(1800.0), sigma=0.75, size=cross_border_mask.sum())

    return np.round(np.clip(distances, 0.0, 12000.0), 2)

In [8]:
def generate_everyday_transactions(customers, n_transactions, start, n_days, seed=random_seed):
    rng = np.random.default_rng(seed)

    customer_weights = customers["activity_level"].to_numpy()
    customer_weights = customer_weights / customer_weights.sum()

    selected_customer_positions = rng.choice(
        customers.index.to_numpy(),
        size=n_transactions,
        replace=True,
        p=customer_weights,
    )

    selected_customers = customers.loc[selected_customer_positions].reset_index(drop=True)

    timestamps = sample_normal_timestamps(start, n_days, n_transactions, rng)

    merchant_categories = rng.choice(
        MERCHANT_CATEGORIES,
        size=n_transactions,
        p=MERCHANT_CATEGORY_PROBABILITIES,
    )

    transaction_channels = choose_transaction_channels(merchant_categories, rng)
    device_types = choose_device_types(selected_customers["primary_device_type"].to_numpy(), rng)
    amounts = sample_normal_amounts(merchant_categories, selected_customers["normal_spend_level"].to_numpy(), rng)
    is_cross_border = sample_normal_cross_border_flags(merchant_categories, transaction_channels, rng)

    merchant_countries = []
    merchant_ids = []

    for home_country, merchant_category, cross_border in zip(
        selected_customers["home_country"],
        merchant_categories,
        is_cross_border,
    ):
        if cross_border:
            merchant_country = choose_other_country(home_country, rng)
        else:
            merchant_country = home_country

        merchant_countries.append(merchant_country)
        merchant_ids.append(make_merchant_id(merchant_category, merchant_country, rng))

    account_age_days = [
        get_account_age_days(timestamp, signup_date)
        for timestamp, signup_date in zip(timestamps, selected_customers["signup_date"])
    ]

    distances = sample_distances_from_home(
        selected_customers["normal_travel_radius"].to_numpy(),
        is_cross_border,
        transaction_channels,
        rng,
    )

    transactions = pd.DataFrame({
        "timestamp": timestamps,
        "customer_id": selected_customers["customer_id"],
        "account_age_days": account_age_days,
        "home_country": selected_customers["home_country"],
        "merchant_id": merchant_ids,
        "merchant_country": merchant_countries,
        "merchant_category": merchant_categories,
        "device_type": device_types,
        "transaction_channel": transaction_channels,
        "amount": amounts,
        "distance_from_home": distances,
        "is_cross_border": is_cross_border,
        "is_subscription": False,
        "subscription_name": "none",
        "is_fraud": 0,
        "fraud_source": "none",
        "fraud_scenario": "none",
    })

    return transactions

In [9]:
def choose_subscription_channel(subscription_name, rng):
    credit_card_probability = 0.18

    if subscription_name in ["Netflix", "Spotify", "Videoland"]:
        credit_card_probability = 0.32

    if subscription_name in ["NS Flex", "Swapfiets", "Basic-Fit"]:
        credit_card_probability = 0.10

    if rng.random() < credit_card_probability:
        return "credit_card_online"

    return "ideal_online"


def generate_subscription_transactions(customers, start, n_days, seed=random_seed):
    rng = np.random.default_rng(seed)
    rows = []

    start_timestamp = pd.Timestamp(start)
    end_timestamp = start_timestamp + pd.to_timedelta(n_days, unit="D")

    subscription_counts = rng.choice(
        [0, 1, 2, 3],
        size=len(customers),
        p=[0.55, 0.30, 0.12, 0.03],
    )

    for customer_position, subscription_count in enumerate(subscription_counts):
        if subscription_count == 0:
            continue

        customer = customers.iloc[customer_position]
        subscriptions = rng.choice(
            len(SUBSCRIPTION_PRODUCTS),
            size=subscription_count,
            replace=False,
        )

        for subscription_index in subscriptions:
            subscription = SUBSCRIPTION_PRODUCTS[subscription_index]

            charge_day = rng.integers(1, 29)
            first_month_offset = rng.integers(0, 4)
            charge_date = start_timestamp + pd.DateOffset(months=int(first_month_offset))
            charge_date = charge_date.replace(day=int(charge_day))

            while charge_date < end_timestamp:
                charge_time = (
                    charge_date
                    + pd.to_timedelta(rng.normal(9.0, 1.5), unit="h")
                    + pd.to_timedelta(rng.integers(0, 60), unit="m")
                    + pd.to_timedelta(rng.integers(0, 60), unit="s")
                )

                is_cross_border = rng.random() < 0.06

                if is_cross_border:
                    merchant_country = choose_other_country(customer["home_country"], rng)
                else:
                    merchant_country = customer["home_country"]

                amount = subscription["base_amount"] * rng.normal(1.0, 0.03)
                amount = round(max(1.0, amount), 2)

                distance_from_home = 0.0

                if is_cross_border:
                    distance_from_home = rng.lognormal(mean=np.log(1600.0), sigma=0.6)

                rows.append({
                    "timestamp": charge_time,
                    "customer_id": customer["customer_id"],
                    "account_age_days": get_account_age_days(charge_time, customer["signup_date"]),
                    "home_country": customer["home_country"],
                    "merchant_id": f"{merchant_country}_SUB_{subscription['subscription_name'].upper().replace(' ', '_')}",
                    "merchant_country": merchant_country,
                    "merchant_category": subscription["merchant_category"],
                    "device_type": customer["primary_device_type"],
                    "transaction_channel": choose_subscription_channel(subscription["subscription_name"], rng),
                    "amount": amount,
                    "distance_from_home": round(distance_from_home, 2),
                    "is_cross_border": bool(is_cross_border),
                    "is_subscription": True,
                    "subscription_name": subscription["subscription_name"],
                    "is_fraud": 0,
                    "fraud_source": "none",
                    "fraud_scenario": "none",
                })

                charge_date = charge_date + pd.DateOffset(months=1)

    return pd.DataFrame(rows)

In [10]:
def choose_fraud_source(rng):
    return rng.choice(
        ["account_takeover", "stolen_card", "new_account_fraud"],
        p=[0.42, 0.38, 0.20],
    )


def choose_fraud_device_type(primary_device_type, rng):
    if rng.random() < 0.72:
        other_devices = [device_type for device_type in DEVICE_TYPES if device_type != primary_device_type]
        return rng.choice(other_devices)

    return primary_device_type


def choose_existing_fraud_customer(customers, fraud_source, rng):
    if fraud_source == "account_takeover":
        older_customers = customers[customers["signup_date"] < customers["signup_date"].quantile(0.65)]
        return older_customers.sample(1, random_state=int(rng.integers(0, 1_000_000))).iloc[0]

    customer_weights = customers["activity_level"].to_numpy()
    customer_weights = customer_weights / customer_weights.sum()
    customer_position = rng.choice(customers.index.to_numpy(), p=customer_weights)

    return customers.loc[customer_position]


def make_new_fraud_customer(timestamp, event_number, rng):
    signup_date = pd.Timestamp(timestamp).normalize() - pd.to_timedelta(rng.integers(0, 15), unit="D")

    return pd.Series({
        "customer_id": f"FRAUD_CUST_{event_number:05d}",
        "signup_date": signup_date,
        "home_country": rng.choice(COUNTRIES, p=COUNTRY_PROBABILITIES),
        "primary_device_type": rng.choice(DEVICE_TYPES, p=FRAUD_DEVICE_TYPE_PROBABILITIES),
        "normal_spend_level": rng.lognormal(mean=0.0, sigma=0.45),
        "normal_travel_radius": rng.gamma(shape=2.0, scale=5.0) + 1.0,
        "activity_level": 1.0,
    })


def get_fraud_customer(customers, fraud_source, timestamp, event_number, rng):
    if fraud_source == "new_account_fraud":
        return make_new_fraud_customer(timestamp, event_number, rng)

    return choose_existing_fraud_customer(customers, fraud_source, rng)


def sample_fraud_distance(is_cross_border, minimum_distance, rng):
    if is_cross_border:
        distance = rng.lognormal(mean=np.log(2200.0), sigma=0.7)
    else:
        distance = minimum_distance + rng.exponential(scale=120.0)

    return round(float(np.clip(distance, minimum_distance, 12000.0)), 2)


def make_fraud_row(
    timestamp,
    customer,
    merchant_id,
    merchant_country,
    merchant_category,
    device_type,
    transaction_channel,
    amount,
    distance_from_home,
    is_cross_border,
    fraud_source,
    fraud_scenario,
):
    return {
        "timestamp": timestamp,
        "customer_id": customer["customer_id"],
        "account_age_days": get_account_age_days(timestamp, customer["signup_date"]),
        "home_country": customer["home_country"],
        "merchant_id": merchant_id,
        "merchant_country": merchant_country,
        "merchant_category": merchant_category,
        "device_type": device_type,
        "transaction_channel": transaction_channel,
        "amount": round(float(amount), 2),
        "distance_from_home": distance_from_home,
        "is_cross_border": bool(is_cross_border),
        "is_subscription": False,
        "subscription_name": "none",
        "is_fraud": 1,
        "fraud_source": fraud_source,
        "fraud_scenario": fraud_scenario,
    }

In [11]:
def generate_ping_fraud(customers, start, n_days, event_number, max_transactions, rng):
    n_transactions = int(min(max_transactions, rng.integers(3, 6)))

    base_timestamp = sample_late_night_timestamp(start, n_days, rng)
    fraud_source = choose_fraud_source(rng)
    customer = get_fraud_customer(customers, fraud_source, base_timestamp, event_number, rng)

    device_type = choose_fraud_device_type(customer["primary_device_type"], rng)
    is_cross_border = rng.random() < 0.68

    if is_cross_border:
        merchant_country = choose_other_country(customer["home_country"], rng)
    else:
        merchant_country = customer["home_country"]

    merchant_category = "Digital Goods"
    merchant_id = make_merchant_id(merchant_category, merchant_country, rng)
    distance_from_home = sample_fraud_distance(is_cross_border, 5.0, rng)

    rows = []

    for transaction_number in range(n_transactions):
        timestamp = base_timestamp + pd.to_timedelta(rng.integers(10, 180) * transaction_number, unit="s")
        amount = rng.uniform(0.10, 1.99)

        rows.append(make_fraud_row(
            timestamp=timestamp,
            customer=customer,
            merchant_id=merchant_id,
            merchant_country=merchant_country,
            merchant_category=merchant_category,
            device_type=device_type,
            transaction_channel="credit_card_online",
            amount=amount,
            distance_from_home=distance_from_home,
            is_cross_border=is_cross_border,
            fraud_source=fraud_source,
            fraud_scenario="ping",
        ))

    return rows


def generate_high_value_fraud(customers, start, n_days, high_amount_threshold, event_number, rng):
    if rng.random() < 0.35:
        timestamp = sample_late_night_timestamp(start, n_days, rng)
    else:
        timestamp = sample_normal_timestamps(start, n_days, 1, rng)[0]

    fraud_source = choose_fraud_source(rng)
    customer = get_fraud_customer(customers, fraud_source, timestamp, event_number, rng)

    merchant_category = rng.choice(["Electronics", "Jewelry", "Luxury Retail", "Travel"], p=[0.38, 0.24, 0.20, 0.18])
    is_cross_border = rng.random() < 0.74

    if is_cross_border:
        merchant_country = choose_other_country(customer["home_country"], rng)
    else:
        merchant_country = customer["home_country"]

    merchant_id = make_merchant_id(merchant_category, merchant_country, rng)
    device_type = choose_fraud_device_type(customer["primary_device_type"], rng)
    transaction_channel = rng.choice(
		["credit_card_online", "ideal_online", "pos_pin"],
		p=[0.58, 0.12, 0.30],
	)
    amount = rng.lognormal(mean=np.log(high_amount_threshold * 1.7), sigma=0.45)
    distance_from_home = sample_fraud_distance(is_cross_border, 80.0, rng)

    return [make_fraud_row(
        timestamp=timestamp,
        customer=customer,
        merchant_id=merchant_id,
        merchant_country=merchant_country,
        merchant_category=merchant_category,
        device_type=device_type,
        transaction_channel=transaction_channel,
        amount=amount,
        distance_from_home=distance_from_home,
        is_cross_border=is_cross_border,
        fraud_source=fraud_source,
        fraud_scenario="high_value_extrication",
    )]


def generate_velocity_fraud(customers, start, n_days, event_number, max_transactions, rng):
    n_transactions = int(min(max_transactions, rng.integers(10, 19)))

    if n_transactions < 10:
        return []

    base_timestamp = sample_late_night_timestamp(start, n_days, rng)

    fraud_source = choose_fraud_source(rng)
    customer = get_fraud_customer(customers, fraud_source, base_timestamp, event_number, rng)

    merchant_category = rng.choice(["Retail", "Electronics", "Digital Goods", "Food & Dining"], p=[0.34, 0.26, 0.25, 0.15])
    is_cross_border = rng.random() < 0.57

    if is_cross_border:
        merchant_country = choose_other_country(customer["home_country"], rng)
    else:
        merchant_country = customer["home_country"]

    merchant_id = make_merchant_id(merchant_category, merchant_country, rng)
    device_type = choose_fraud_device_type(customer["primary_device_type"], rng)
    transaction_channel = rng.choice(
		["credit_card_online", "ideal_online", "pos_pin"],
		p=[0.55, 0.30, 0.15],
	)
    distance_from_home = sample_fraud_distance(is_cross_border, 25.0, rng)

    offsets = np.sort(rng.integers(0, 300, size=n_transactions))
    rows = []

    for offset_seconds in offsets:
        timestamp = base_timestamp + pd.to_timedelta(int(offset_seconds), unit="s")
        amount = rng.lognormal(mean=np.log(65.0), sigma=0.55)
        amount = np.clip(amount, 12.0, 250.0)

        rows.append(make_fraud_row(
            timestamp=timestamp,
            customer=customer,
            merchant_id=merchant_id,
            merchant_country=merchant_country,
            merchant_category=merchant_category,
            device_type=device_type,
            transaction_channel=transaction_channel,
            amount=amount,
            distance_from_home=distance_from_home,
            is_cross_border=is_cross_border,
            fraud_source=fraud_source,
            fraud_scenario="velocity_attack",
        ))

    return rows

In [12]:
def generate_fraud_transactions(customers, normal_transactions, start, n_days, target_fraud_count, seed=random_seed):
    rng = np.random.default_rng(seed)
    rows = []
    event_number = 1

    high_amount_threshold = normal_transactions["amount"].quantile(0.99)

    while len(rows) < target_fraud_count:
        remaining_transactions = target_fraud_count - len(rows)

        if remaining_transactions < 3:
            scenario = "high_value_extrication"
        elif remaining_transactions < 10:
            scenario = rng.choice(["ping", "high_value_extrication"], p=[0.55, 0.45])
        else:
            scenario = rng.choice(
                ["ping", "high_value_extrication", "velocity_attack"],
                p=[0.34, 0.36, 0.30],
            )

        if scenario == "ping":
            new_rows = generate_ping_fraud(
                customers=customers,
                start=start,
                n_days=n_days,
                event_number=event_number,
                max_transactions=remaining_transactions,
                rng=rng,
            )

        elif scenario == "high_value_extrication":
            new_rows = generate_high_value_fraud(
                customers=customers,
                start=start,
                n_days=n_days,
                high_amount_threshold=high_amount_threshold,
                event_number=event_number,
                rng=rng,
            )

        else:
            new_rows = generate_velocity_fraud(
                customers=customers,
                start=start,
                n_days=n_days,
                event_number=event_number,
                max_transactions=remaining_transactions,
                rng=rng,
            )

        rows.extend(new_rows)
        event_number += 1

    return pd.DataFrame(rows[:target_fraud_count])

In [13]:
def round_and_order_columns(df):
    df = df.copy()

    string_columns = [
        "customer_id",
        "home_country",
        "merchant_id",
        "merchant_country",
        "merchant_category",
        "device_type",
        "transaction_channel",
        "subscription_name",
        "fraud_source",
        "fraud_scenario",
    ]

    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["account_age_days"] = df["account_age_days"].astype(int)
    df["amount"] = df["amount"].round(3)
    df["distance_from_home"] = df["distance_from_home"].round(3)
    df["is_cross_border"] = df["is_cross_border"].astype(bool)
    df["is_subscription"] = df["is_subscription"].astype(bool)
    df["is_fraud"] = df["is_fraud"].astype(int)

    for column in string_columns:
        df[column] = df[column].map(str)

    return df[OUTPUT_COLUMNS]


def simulate_fraud_dataset(dataset_config, seed=random_seed):
    customers = make_customers(
        n_customers=dataset_config["n_customers"],
        start=dataset_config["start"],
        seed=seed,
    )

    everyday_transactions = generate_everyday_transactions(
        customers=customers,
        n_transactions=dataset_config["n_everyday_transactions"],
        start=dataset_config["start"],
        n_days=dataset_config["n_days"],
        seed=seed + 1,
    )

    subscription_transactions = generate_subscription_transactions(
        customers=customers,
        start=dataset_config["start"],
        n_days=dataset_config["n_days"],
        seed=seed + 2,
    )

    normal_transactions = pd.concat(
        [everyday_transactions, subscription_transactions],
        ignore_index=True,
    )

    target_fraud_count = int(
        round(
            len(normal_transactions)
            * dataset_config["target_fraud_rate"]
            / (1.0 - dataset_config["target_fraud_rate"])
        )
    )

    fraud_transactions = generate_fraud_transactions(
        customers=customers,
        normal_transactions=normal_transactions,
        start=dataset_config["start"],
        n_days=dataset_config["n_days"],
        target_fraud_count=target_fraud_count,
        seed=seed + 3,
    )

    df = pd.concat(
        [normal_transactions, fraud_transactions],
        ignore_index=True,
    )

    df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    df.insert(0, "transaction_id", make_transaction_ids(len(df), seed=seed))

    df = round_and_order_columns(df)

    return df, customers

In [14]:
dataset_config = {
    "start": "2025-01-01",
    "n_days": 365,
    "n_customers": 5_000,
    "n_everyday_transactions": 100_000,
    "target_fraud_rate": 0.01,
}

In [15]:
df, customers = simulate_fraud_dataset(dataset_config, seed=random_seed)

os.makedirs("../generated_data/tabular", exist_ok=True)

df.to_parquet("../generated_data/tabular/imbalanced_fraud_detection.parquet", index=False)
customers.to_parquet("../generated_data/tabular/fraud_customer_profiles.parquet", index=False)

In [16]:
df.head()

,transaction_id,timestamp,customer_id,account_age_days,home_country,merchant_id,merchant_country,merchant_category,device_type,transaction_channel,amount,distance_from_home,is_cross_border,is_subscription,subscription_name,is_fraud,fraud_source,fraud_scenario
0,76601c32-dc5e-51c4-99a3-f53b65b6a964,2025-11-04 09:54:11.975329849,CUST_03962,766,NL,NL_SUB_CLOUD_STORAGE,NL,Digital Goods,laptop,ideal_online,9.83,0.00,False,True,Cloud Storage,0,none,none
1,b9a18927-fe67-5549-9723-53b4db674102,2025-08-27 09:00:31.943673104,CUST_03298,3154,NL,NL_RETAIL_025,NL,Retail,desktop,ideal_online,27.72,20.43,False,False,none,0,none,none
2,e63f9d10-ae62-5063-a3f1-06d05d3921f8,2025-01-22 19:47:19.868015544,CUST_03796,2417,GB,FR_RETAIL_175,FR,Retail,iphone,credit_card_online,67.91,1038.59,True,False,none,0,none,none
3,0aec4b43-86a8-560d-a21e-ef4f009bb9ea,2025-05-02 14:40:27.971042380,CUST_03369,664,BE,BE_GROCERIES_088,BE,Groceries,iphone,pos_pin,18.44,7.17,False,False,none,0,none,none
4,4bbe2a05-7f08-5d28-bf26-178981c8ec82,2025-09-01 12:25:05.506817336,CUST_02586,3274,DE,GB_ELECTRONICS_009,GB,Electronics,iphone,pos_pin,129.54,1575.31,True,False,none,0,none,none


In [17]:
df["is_fraud"].mean().round(3)

np.float64(0.01)

In [18]:
df["fraud_scenario"].value_counts()

fraud_scenario
none                      133529
velocity_attack              954
ping                         297
high_value_extrication        98
Name: count, dtype: int64

In [19]:
df.groupby("is_fraud")["amount"].describe().round(3)

,count,mean,std,min,25%,50%,75%,max
is_fraud,,,,,,,,
0,133529.0,41.912,72.211,1.0,10.91,20.45,45.52,3155.21
1,1349.0,97.051,161.441,0.1,25.44,55.06,95.44,1295.18


In [20]:
df.groupby("is_fraud")[["is_cross_border", "is_subscription"]].mean().round(3)

,is_cross_border,is_subscription
is_fraud,,
0,0.036,0.251
1,0.609,0.000


In [21]:
df.groupby("fraud_scenario")["amount"].describe().round(3)

,count,mean,std,min,25%,50%,75%,max
fraud_scenario,,,,,,,,
high_value_extrication,98.0,587.426,256.586,206.09,401.785,500.68,732.528,1295.18
none,133529.0,41.912,72.211,1.00,10.910,20.45,45.520,3155.21
ping,297.0,1.040,0.538,0.10,0.550,1.02,1.510,1.98
velocity_attack,954.0,76.567,46.301,12.00,45.760,63.21,95.448,250.00


In [22]:
(df.groupby("fraud_scenario")["amount"].describe()["count"].sum() - 133529.0)/134878.0

np.float64(0.010001631103664052)

In [23]:
df[df["is_subscription"]].head()

,transaction_id,timestamp,customer_id,account_age_days,home_country,merchant_id,merchant_country,merchant_category,device_type,transaction_channel,amount,distance_from_home,is_cross_border,is_subscription,subscription_name,is_fraud,fraud_source,fraud_scenario
0,76601c32-dc5e-51c4-99a3-f53b65b6a964,2025-11-04 09:54:11.975329849,CUST_03962,766,NL,NL_SUB_CLOUD_STORAGE,NL,Digital Goods,laptop,ideal_online,9.83,0.0,False,True,Cloud Storage,0,none,none
9,3153f5e7-ee59-58b4-95c8-c8fa3131e549,2025-09-02 08:25:38.839076313,CUST_00864,3753,NL,NL_SUB_SWAPFIETS,NL,Bike & Mobility,macbook,credit_card_online,20.45,0.0,False,True,Swapfiets,0,none,none
10,5a271ae6-4d63-543a-90d2-f0d08e6540e1,2025-06-14 08:46:20.053123277,CUST_02509,2765,NL,NL_SUB_SPOTIFY,NL,Entertainment,iphone,ideal_online,10.90,0.0,False,True,Spotify,0,none,none
13,1620ee3b-0ab7-5517-b383-5e3634ffedf1,2025-07-01 10:16:10.622047634,CUST_00038,3278,GB,GB_SUB_MEAL_KIT,GB,Food & Dining,iphone,ideal_online,59.95,0.0,False,True,Meal Kit,0,none,none
16,3a642a77-086e-5afe-9b2a-6c1a57c46bef,2025-04-18 09:49:33.021341260,CUST_01969,1604,NL,NL_SUB_CLOUD_STORAGE,NL,Digital Goods,android,ideal_online,9.27,0.0,False,True,Cloud Storage,0,none,none


In [24]:
df["transaction_channel"].value_counts(normalize=True).round(12)

transaction_channel
pos_pin               0.481027
ideal_online          0.408651
credit_card_online    0.095294
atm_cash              0.015028
Name: proportion, dtype: float64

In [25]:
df["merchant_category"].value_counts(normalize=True).round(12).sum()

np.float64(1.0)